# Environment Verification
Run all cells top to bottom. Every cell should print ✓ with no errors.

**Make sure you selected the `VFR Project (3.10)` kernel before running.**

In [1]:
# Cell 1 — Core packages
import sys
import numpy as np
import cv2
import mediapipe as mp
from PIL import Image
import matplotlib
import scipy
import sklearn

print(f"Python    : {sys.version.split()[0]}")
print(f"NumPy     : {np.__version__}")
print(f"OpenCV    : {cv2.__version__}")
print(f"MediaPipe : {mp.__version__}")
print(f"Pillow    : {Image.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print("✓ Core packages OK")

Python    : 3.10.20
NumPy     : 2.2.6
OpenCV    : 4.13.0
MediaPipe : 0.10.32
Pillow    : 12.0.0
Matplotlib: 3.10.8
✓ Core packages OK


In [2]:
# Cell 2 — PyTorch + CUDA
import torch
import torchvision

print(f"PyTorch       : {torch.__version__}")
print(f"Torchvision   : {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA device   : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version  : {torch.version.cuda}")
    # Quick GPU tensor test
    x = torch.randn(3, 3).cuda()
    print(f"GPU tensor    : {x.device} ✓")
else:
    print("Running on CPU (expected on local machine)")

print("✓ PyTorch OK")

PyTorch       : 2.10.0
Torchvision   : 0.25.0
CUDA available: False
Running on CPU (expected on local machine)
✓ PyTorch OK


In [3]:
# Cell 3 — GAN / vision dependencies
import kornia
import einops

print(f"Kornia : {kornia.__version__}")
print(f"Einops : {einops.__version__}")

# Quick kornia smoke test — TPS will be used for garment warping
img = torch.rand(1, 3, 64, 64)
gray = kornia.color.rgb_to_grayscale(img)
assert gray.shape == (1, 1, 64, 64)
print("Kornia transform test ✓")
print("✓ GAN dependencies OK")

Kornia : 0.8.2
Einops : 0.8.2
Kornia transform test ✓
✓ GAN dependencies OK


In [4]:
# Cell 4 — Segment Anything Model
from segment_anything import sam_model_registry, SamPredictor
import os

print("SAM import OK")
checkpoint = "checkpoints/sam_vit_h_4b8939.pth"
if os.path.exists(checkpoint):
    size_gb = os.path.getsize(checkpoint) / 1e9
    print(f"SAM checkpoint found: {size_gb:.1f} GB ✓")
else:
    print("⚠ SAM checkpoint not found at checkpoints/sam_vit_h_4b8939.pth")
    print("  Run the setup script to download it, or download manually from:")
    print("  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth")

print("✓ SAM OK")

SAM import OK
⚠ SAM checkpoint not found at checkpoints/sam_vit_h_4b8939.pth
  Run the setup script to download it, or download manually from:
  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
✓ SAM OK


In [5]:
# Cell 5 — Evaluation metrics
import lpips
from skimage.metrics import structural_similarity

# Quick SSIM test
a = np.random.rand(64, 64)
b = np.random.rand(64, 64)
score = structural_similarity(a, b, data_range=1.0)
print(f"SSIM test score: {score:.4f} (random, expected ~0)")

# Quick LPIPS init
loss_fn = lpips.LPIPS(net='vgg', verbose=False)
print(f"LPIPS model loaded ✓")
print("✓ Evaluation metrics OK")

SSIM test score: 0.0089 (random, expected ~0)


/Users/purveshg/anaconda3/envs/vfr-env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/purveshg/anaconda3/envs/vfr-env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /Users/purveshg/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|████████████████████████████████████████| 528M/528M [00:44<00:00, 12.4MB/s]


LPIPS model loaded ✓
✓ Evaluation metrics OK


In [6]:
# Cell 6 — MediaPipe pose smoke test
mp_pose = mp.solutions.pose

# Create a blank image and run pose on it (no person = no landmarks, but no crash)
dummy_img = np.zeros((480, 360, 3), dtype=np.uint8)
with mp_pose.Pose(static_image_mode=True, model_complexity=0) as pose:
    results = pose.process(dummy_img)
    detected = results.pose_landmarks is not None
    print(f"Pose on blank image — landmarks detected: {detected} (expected False)")

print("✓ MediaPipe pose OK")

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [ ]:
# Cell 7 — Folder structure check
from pathlib import Path

expected_dirs = [
    "data/viton_hd/train/image",
    "data/viton_hd/train/cloth",
    "data/viton_hd/test/image",
    "outputs/pose",
    "outputs/segmentation",
    "outputs/warping",
    "outputs/tryon",
    "checkpoints",
    "src/pose",
    "src/segmentation",
]

all_ok = True
for d in expected_dirs:
    exists = Path(d).exists()
    status = '✓' if exists else '✗ MISSING'
    print(f"  {status}  {d}")
    if not exists:
        all_ok = False

if not all_ok:
    print("\n⚠ Some folders are missing. Run setup_env.sh to create them.")
else:
    print("\n✓ All folders present")

In [ ]:
# Cell 8 — Final summary
print("="*50)
print("  ENVIRONMENT SUMMARY")
print("="*50)
print(f"  Python  : {sys.version.split()[0]}")
print(f"  PyTorch : {torch.__version__}")
print(f"  CUDA    : {torch.cuda.is_available()}")
print(f"  Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print()
print("  All checks passed. You are ready to start.")
print("  Open 01_pose_estimation.ipynb to begin.")
print("="*50)